# 04 – Final model and test-set evaluation

Finalising the model chosen in `03_models_and_evaluation.ipynb`: choosing the decision threshold, checking one improvement and a time-ordered split, and then evaluating once on the held-out test set.

## 1. Setup

In [1]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import fbeta_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

SEED = 42
TEST_SIZE = 0.2
N_FOLDS = 5

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.13.11
pandas: 3.0.6
scikit-learn: 1.9.1


In [2]:
df = pd.read_csv("../data/ai4i2020.csv")

target = "Machine failure"
drop_cols = ["UDI", "Product ID", "TWF", "HDF", "PWF", "OSF", "RNF"]

X = df.drop(columns=drop_cols + [target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)

numeric_cols = ["Air temperature [K]", "Process temperature [K]",
                "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]
categorical_cols = ["Type"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])

cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Settings selected by the grid search in notebook 03, Section 4
final_rf = RandomForestClassifier(
    n_estimators=500, max_depth=None, min_samples_leaf=5, max_features=0.5,
    class_weight="balanced_subsample", random_state=SEED, n_jobs=-1,
)
final_pipe = Pipeline([("prep", preprocessor), ("model", final_rf)])

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (8000, 6) Test: (2000, 6)


## 2. Decision threshold

The model outputs a failure probability, and the threshold decides when that counts as a failure. I compare thresholds using out-of-fold probabilities on the training set only, and pick the one with the highest F2. The test set is not used.

In [3]:
oof_prob = cross_val_predict(final_pipe, X_train, y_train, cv=cv, method="predict_proba")[:, 1]

thresholds = np.round(np.arange(0.10, 0.91, 0.05), 2)
rows = []
for t in thresholds:
    pred = (oof_prob >= t).astype(int)
    rows.append({
        "threshold": t,
        "f2": fbeta_score(y_train, pred, beta=2, zero_division=0),
        "precision": precision_score(y_train, pred, zero_division=0),
        "recall": recall_score(y_train, pred, zero_division=0),
        "flagged": int(pred.sum()),
    })

threshold_table = pd.DataFrame(rows).set_index("threshold").round(3)
best_threshold = threshold_table["f2"].idxmax()

print("Best threshold by out-of-fold F2:", best_threshold)
threshold_table

Best threshold by out-of-fold F2: 0.5


,f2,precision,recall,flagged
threshold,,,,
0.10,0.612,0.251,0.956,1033
0.15,0.648,0.293,0.930,860
0.20,0.675,0.334,0.908,737
0.25,0.697,0.371,0.893,652
0.30,0.715,0.417,0.871,566
0.35,0.731,0.467,0.852,495
0.40,0.737,0.520,0.823,429
0.45,0.740,0.574,0.797,376
0.50,0.748,0.647,0.779,326


**Findings:**
- The best threshold is 0.5 (F2 0.748), so the default turns out to be the right choice. The class weighting in the forest has already shifted its probabilities towards the failure class.
- F2 is fairly flat between 0.40 and 0.55 (0.737 to 0.748), so the exact choice matters little in that range.
- Lowering the threshold catches more failures but raises many more alarms. At 0.30, recall rises to 0.871 but the model flags 566 rows instead of 326, and precision falls to 0.417.
- I will keep 0.5. If missed failures turned out to be much more costly in practice, a lower threshold such as 0.30 would be a reasonable alternative.